# JustBuild Foundation — Recruitment Agent

One job description, 25 resumes, one shortlist. This notebook runs the **three-agent** design you drew on the board:

**Input → Agent 1 (Ingest & Extract) → Agent 2 (Judge & Prioritise) → Agent 3 (Communicate) → You (the human)**

- **Agent 1** turns messy resumes into a clean table of facts.
- **Agent 2** scores each candidate against the rubric *with evidence*, applies the hard rule **in code**, and ranks them.
- **Agent 3** drafts a recommendation to the hiring manager. It does **not** send anything — that stays your call.

Everything the notebook needs — the resumes, the job description, and the **OC Brain** — is pulled from the hosted pack. Nothing is pasted inline.

> Run the cells top to bottom. Watch for the green checkmarks.

In [ ]:
#@title 1 · Setup  { display-mode: "form" }
# Installs the few libraries we need and confirms they loaded.
!pip -q install google-generativeai pypdf requests pandas >/dev/null 2>&1
import requests, io, re, json, time
from pypdf import PdfReader
import pandas as pd
print('Setup complete - libraries ready.')

In [ ]:
#@title 2 · Configuration  { display-mode: "form" }
# 'Live Gemini' calls the real model (needs a key). 'Switch to OC Brain' skips the
# network and uses the prepared results.
import getpass

MODE = "Live Gemini"  #@param ["Live Gemini", "Switch to OC Brain"]
MODEL = "gemini-2.5-flash-lite"  #@param {type:"string"}

# Inputs and the OC Brain live here (hosted, not pasted into the notebook):
BASE = "https://justbuild.orangecaterpillar.com/foundation-material/recruitment"
RESUMES_URL  = f"{BASE}/Resumes.pdf"
JD_URL       = f"{BASE}/Job_Description.pdf"
OC_BRAIN_URL = f"{BASE}/oc-brain.json"

# API key is only needed for Live Gemini. Typed in hidden; never saved in the notebook.
GEMINI_API_KEY = ""
if MODE == "Live Gemini":
    GEMINI_API_KEY = getpass.getpass('Paste your Gemini API key (hidden). Leave blank to use the OC Brain: ').strip()

print(f'Mode: {MODE}   Model: {MODEL}')

In [ ]:
#@title 3 · Load the inputs and the OC Brain  { display-mode: "form" }
# Pull the resumes and the job description straight from the hosted pack, and split
# the resume PDF into one block of text per candidate. No data is inlined here.

def fetch_pdf_pages(url):
    r = requests.get(url, timeout=30); r.raise_for_status()
    reader = PdfReader(io.BytesIO(r.content))
    return [(p.extract_text() or '') for p in reader.pages]

resume_pages = fetch_pdf_pages(RESUMES_URL)
RESUMES = []
for t in resume_pages:
    if re.search(r'Candidate \d+ of \d+', t):
        RESUMES.append(re.sub(r'Candidate \d+ of \d+\s*', '', t).strip())

JD_TEXT = '\n'.join(fetch_pdf_pages(JD_URL))
print(f'Loaded {len(RESUMES)} candidate resumes and the job description from the hosted pack.')

# The OC Brain is fetched too - a prepared results file the notebook CALLS, not baked
# into the code. It stands ready if the live model stalls.
OC_BRAIN = requests.get(OC_BRAIN_URL, timeout=30).json()
print(f"OC Brain on standby - {len(OC_BRAIN['candidates'])} prepared results loaded.")

In [ ]:
#@title 4 · The engine (and the OC Brain safety net)  { display-mode: "form" }
# Decide the run mode, wire up Gemini if needed, and define one careful call that
# retries on the ordinary errors (429 busy/quota, 503 overloaded).

USE_OC_BRAIN = (MODE == 'Switch to OC Brain') or (not GEMINI_API_KEY)
if USE_OC_BRAIN and MODE == 'Live Gemini':
    print('No API key given - running on the OC Brain.')

model = None
if not USE_OC_BRAIN:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(MODEL)

def call_gemini(prompt, tries=4):
    """Call the model, backing off on the errors you'll actually meet."""
    last = None
    for i in range(tries):
        try:
            return model.generate_content(prompt).text
        except Exception as e:
            last = e; msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg: what = '429 (busy / free-tier limit)'
            elif '503' in msg or 'UNAVAILABLE' in msg:      what = '503 (model overloaded)'
            elif '404' in msg:                              what = '404 (model name)'
            else:                                           what = 'an error'
            wait = 2 ** i
            print(f'Gemini returned {what} - try {i+1}/{tries}. Waiting {wait}s...')
            time.sleep(wait)
    raise last

def extract_json(text):
    """Pull a JSON array/object out of the model's reply."""
    text = text.strip()
    text = re.sub(r'^```(?:json)?', '', text).strip()
    text = re.sub(r'```$', '', text).strip()
    start = min([i for i in (text.find('['), text.find('{')) if i != -1])
    return json.loads(text[start:])

def switch_to_oc_brain(reason):
    global USE_OC_BRAIN
    print(f'{reason} Switching to the OC Brain to keep going.')
    USE_OC_BRAIN = True

print('Engine ready.', '(OC Brain)' if USE_OC_BRAIN else f'(Live: {MODEL})')

## Agent 1 — Ingest & Extract
Turn 25 messy resumes into one clean table of facts. Live, Gemini reads each resume; on the OC Brain, the same facts come from the prepared file.

In [ ]:
#@title 5 · Agent 1 runs  { display-mode: "form" }
FIELDS = ['python_years', 'backend_python_years', 'cloud', 'relational_db', 'rest_api']
records = {}  # name -> merged record across the three agents

if not USE_OC_BRAIN:
    try:
        numbered = '\n\n'.join(f'--- RESUME {i+1} ---\n{t}' for i, t in enumerate(RESUMES))
        prompt = (
            'You are Agent 1 in a recruitment pipeline. For EACH resume below, extract exactly these '
            'fields and return ONLY a JSON array (no prose):\n'
            '  name (string), title (string), python_years (integer backend Python years; 0 if none), '
            'backend_python_years (integer; 0 if their Python is not backend, e.g. test/data/automation), '
            'cloud ("AWS"/"GCP"/"Azure"/"none"), '
            'relational_db ("PostgreSQL"/"MySQL"/"SQLite"/"MongoDB (NoSQL)"/"none"), '
            'rest_api (true/false).\n\n' + numbered
        )
        for d in extract_json(call_gemini(prompt)):
            records[d['name']] = {'name': d['name'], 'title': d.get('title',''),
                                  'extract': {k: d.get(k) for k in FIELDS}}
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')

if USE_OC_BRAIN:
    for c in OC_BRAIN['candidates']:
        records[c['name']] = {'name': c['name'], 'title': c['title'], 'extract': dict(c['extract'])}

df1 = pd.DataFrame([{'Candidate': r['name'], **r['extract']} for r in records.values()])
print(f'Agent 1 extracted {len(df1)} candidates:')
df1

## Agent 2 — Judge & Prioritise
Score each candidate against the rubric **with evidence**, then apply the hard rule and rank. Notice the hard rule and the weighting live in **code**, not in the model.

In [ ]:
#@title 6 · Agent 2 runs  { display-mode: "form" }
# The rubric and the hard rule are OUR rules, enforced here in code.
WEIGHTS = {'python_depth': 0.35, 'production_impact': 0.30, 'data_and_apis': 0.20, 'stability': 0.15}
HARD_RULE = 'At least 3 years of backend Python AND cloud experience.'

if not USE_OC_BRAIN:
    try:
        facts = [{'name': r['name'], **r['extract']} for r in records.values()]
        prompt = (
            'You are Agent 2. Score each candidate 0-10 on four criteria against this job:\n'
            + JD_TEXT[:1500] + '\n\n'
            'Criteria: python_depth, production_impact, data_and_apis (REST + relational DB), stability.\n'
            'Also give a one-line evidence string, and a flag (string) ONLY if over-qualified, '
            'missing/unclear history, or a career-changer; otherwise flag null.\n'
            'Return ONLY a JSON array: '
            '{name, python_depth, production_impact, data_and_apis, stability, evidence, flag}.\n\n'
            + json.dumps(facts)
        )
        for s in extract_json(call_gemini(prompt)):
            r = records.get(s['name'])
            if r:
                r['scores'] = {k: s.get(k, 0) for k in WEIGHTS}
                r['evidence'] = s.get('evidence', '')
                r['flag'] = s.get('flag') or None
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')

if USE_OC_BRAIN:
    for c in OC_BRAIN['candidates']:
        r = records.get(c['name'])
        if r:
            r['scores'] = dict(c['scores']); r['evidence'] = c['evidence']; r['flag'] = c['flag']

# --- rules in code: weighted total + hard rule + ranking ---
for r in records.values():
    r['total'] = round(sum(r['scores'][k] * WEIGHTS[k] for k in WEIGHTS) * 10, 1)
    ex = r['extract']
    r['hard_rule_pass'] = (int(ex.get('backend_python_years') or 0) >= 3) and (ex.get('cloud') not in (None, 'none'))

ranked = sorted(records.values(), key=lambda r: (-r['total'], -int(r['extract'].get('python_years') or 0), r['name']))
eligible = [r for r in ranked if r['hard_rule_pass'] and not r['flag']]
TOP5 = eligible[:5]
FLAGGED = [r for r in ranked if r['flag']]

show = pd.DataFrame([{'Candidate': r['name'], 'Score': r['total'],
                      'Hard rule': 'pass' if r['hard_rule_pass'] else '-',
                      'Flag': r['flag'] or '', 'Evidence': r['evidence']} for r in ranked])
print('Ranked candidates (rubric + hard rule applied in code):')
show

In [ ]:
#@title    ...and the recommended top 5  { display-mode: "form" }
print('Agent 2 recommends these 5 for interview:\n')
for i, r in enumerate(TOP5, 1):
    print(f"{i}. {r['name']}  -  {r['total']}/100")
    print(f"   {r['evidence']}\n")
print(f'{len(FLAGGED)} candidates were flagged for your judgement (see the table above).')

## Agent 3 — Communicate  (behind a human gate)
Draft a recommendation to the hiring manager. The agent **recommends**; it never sends. You decide.

In [ ]:
#@title 7 · Agent 3 drafts (nothing is sent)  { display-mode: "form" }
draft = None
if not USE_OC_BRAIN:
    try:
        top = [{'name': r['name'], 'title': r.get('title',''), 'score': r['total'], 'evidence': r['evidence']} for r in TOP5]
        prompt = (
            'You are Agent 3. Write a short, professional email to the Hiring Manager recommending these '
            'five candidates for interview for the Backend Engineer (Python & APIs) role. One line of '
            f'reasoning each. Note that {len(FLAGGED)} other candidates were flagged for human review and '
            'not dropped. End by making clear this is a recommendation and nothing has been sent to '
            'candidates. Return only the email text.\n\n' + json.dumps(top)
        )
        draft = call_gemini(prompt)
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')
if USE_OC_BRAIN or draft is None:
    draft = OC_BRAIN['draft']

print('=' * 70)
print(draft)
print('=' * 70)
print('\nHUMAN GATE: this is a draft. The agent has NOT contacted anyone. You decide what happens next.')

In [ ]:
#@title 8 · Your decision  { display-mode: "form" }
DECISION = "Hold"  #@param ["Hold", "Approve (I will send it myself)", "Reject"]
if DECISION.startswith('Approve'):
    print('You approved the shortlist. The agent still sends nothing - you send it yourself, deliberately.')
elif DECISION == 'Reject':
    print('Rejected. Nothing leaves this notebook.')
else:
    print('On hold. Nothing is sent. The decision stays with you.')

## Keep the result
Save the shortlist and the draft to files you can download or paste into your notes, and compare the agent's top 5 with the room's by-hand top 5.

In [ ]:
#@title 9 · Save the shortlist  { display-mode: "form" }
lines = ['# Recruitment shortlist - Backend Engineer (Python & APIs)', '',
         f'Mode: {"OC Brain" if USE_OC_BRAIN else MODEL}', '', '## Recommended top 5', '']
for i, r in enumerate(TOP5, 1):
    lines.append(f"{i}. **{r['name']}** - {r['total']}/100 - {r['evidence']}")
lines += ['', f'## Flagged for a human ({len(FLAGGED)})', '']
for r in FLAGGED:
    lines.append(f"- {r['name']} - {r['flag']}")
lines += ['', '## Draft to the hiring manager', '', '```', draft, '```']
md_out = '\n'.join(lines)

with open('recruitment_shortlist.md', 'w') as f: f.write(md_out)
with open('recruitment_results.json', 'w') as f:
    json.dump({'mode': 'oc_brain' if USE_OC_BRAIN else MODEL,
               'top5': [{'name': r['name'], 'score': r['total'], 'evidence': r['evidence']} for r in TOP5],
               'flagged': [{'name': r['name'], 'flag': r['flag']} for r in FLAGGED],
               'draft': draft}, f, indent=2)

print('Saved: recruitment_shortlist.md and recruitment_results.json (open them from the file panel on the left).')
print('\n----- copy from here -----\n')
print(md_out)
# To download instead, uncomment:
# from google.colab import files; files.download('recruitment_shortlist.md')

## What you just saw
Three narrow agents, chained: extract, judge, communicate, with the rules in **code** and the final call left to a **human**. Swap the inputs for a different pack and the same pipeline serves a different job - which is exactly what you'll do next with your own use-case.